In [1]:
########################## import modules ###############################

import numpy as np
import os
import sys

import netCDF4 as nc
import linecache as lc
import pandas as pd
import time
from scipy import io as sio
import progressbar
import torch
from shutil import copyfile
import json
import torch
import ipdb

# import Src
sys.path.append("Src")
from Py_Fun import Project, Print_test, Te_IRI_obj, network_develop

if not os.path.isdir('Projects/'):
    os.mkdir('Projects')
    
########################## Read Configurations ###############################
#%matplotlib notebook

# Read Config file
with open('Config.json') as json_data_file:
    Config = json.load(json_data_file)

X_select_keys = [key for key in Config['X_select'].keys()]
X_select_values = [value for value in Config['X_select'].values()]

Y_select_keys = [key for key in Config['Y_select'].keys()]
Y_select_values = [value for value in Config['Y_select'].values()]

Stations = [key for key in Config['Test_num'].keys()]
Test_nums = [value for value in Config['Test_num'].values()]

Format_keys = [key for key in Config['Figure_format'].keys()]
Format_values = [value for value in Config['Figure_format'].values()]

# Read Global Coefficients
#with open('Global.json') as Global_coef:
#    Global = json.load(Global_coef)

# Read Project name, ISR datasets and thresholds for preprocessing
Project_name = Config['Names']['Project']
ISR_name = Config['Names']['ISR']
ISR_range = Config['Para_range']

# Read which variable has been picked
index_X = []
index_Y = []


#ipdb.set_trace()

for i in range(len(X_select_keys)):
    if X_select_values[i]:
        #ipdb.set_trace()
        index_X.append(i)
for i in range(len(Y_select_keys)):
    if Y_select_values[i]:
        index_Y.append(i)

########################## Project ###############################

project = Project(Project_name)
project.create()

# remove the project
#project.remove()

Print_test(ISR_range)
time.sleep(1)

The project 'test' does exist. Would you like to overwrite it? (Y/N) n
Keep the project 'test'
no change has been found in Config file
{'NmF2': [10, 15], 'hmF2': [200, 400], 'Month': [0, 12], 'LT': [0, 24], 'VSH': [100, 400], 'Kp': [0, 10], 'F107': [0, 250], 'Altitude': [200, 700], 'Ne': [9.5, 15], 'r': [0.9, 1], 'Latitude': [-90, 90], 'Longitude': [-180, 180], 'Te': [500, 4000]}


In [5]:
num=8

X = []
Y = []
for i in range(num):
    print(['data/XY_'+str(i)+'.mat'])
    temp = []
    #print(sio.loadmat(['data/XY_'+str(i)+'.mat']))
    temp = sio.loadmat('data/XY_'+str(i)+'.mat')
    X_temp = temp['X']
    Y_temp = temp['Y']
    if i==0:
        X = X_temp
        Y = Y_temp
    else:
        X = np.vstack([X, X_temp])
        Y = np.vstack([Y, Y_temp])
    print(X.shape)
    print(Y.shape)

#print(np.argwhere(np.isnan(X)))
#print(~np.isnan(X[7]))

index = (~np.isnan(X[:,7]) & (Y[:,1]>=1e4) & (X[:,0]>200))
X_t = X[index,:]
Y_t = Y[index,1]

#print(np.argwhere(np.isnan(X_temp)).shape)


# Normlisation
X,Y = project.Normlise(X_t.T, Y_t.T)

Y = Y.unsqueeze(0)

_ , _ = project.Dataset_create(X, Y)
#Print_test(Out_train)
#Print_test(Out_test)
time.sleep(.5)

########################## Modelling ##################################
# to make it simple, change index_Y.shape to 1
if Config['Flags']['Modelling']:
    #Print_test(len(index_Y))
    net = project.Modelling(12, 1)
    Print_test(net)
    time.sleep(.5)

    print(X.shape)
    print(Y.shape)

['data/XY_0.mat']
(1000082, 12)
(1000082, 2)
['data/XY_1.mat']
(2000494, 12)
(2000494, 2)
['data/XY_2.mat']
(3000613, 12)
(3000613, 2)
['data/XY_3.mat']
(4000970, 12)
(4000970, 2)
['data/XY_4.mat']
(5001236, 12)
(5001236, 2)
['data/XY_5.mat']
(6001483, 12)
(6001483, 2)
['data/XY_6.mat']
(7001747, 12)
(7001747, 2)
['data/XY_7.mat']
(8002079, 12)
(8002079, 2)
[ 5.14031134e+02 -3.78205070e-02  1.50346143e+00 -8.57373211e-02
 -8.74166892e+00  1.26147331e+02  6.79996026e+00  9.04625882e+01
  1.38921336e+01             nan  5.75467551e+00  1.15131235e+01]
[171.7683548   44.42460182 104.33853626 104.55360046  13.77303777
 154.91197133   8.75204575  27.74083276  11.69764439          nan
   3.47617416   6.87004013]
(12, 5897943)
test
/export/scratch2/andong/Workspace/4D_Ne/Projects/test/Coef/net.pkl
<class 'str'>
True
0.0005
torch.optim.SGD( net.parameters(), lr=0.0005, momentum=0.8, weight_decay=0.0)
SGD (
Parameter Group 0
    dampening: 0
    lr: 0.0005
    momentum: 0.8
    nesterov: False


OSError: [Errno 12] Cannot allocate memory

In [ ]:
########################## Modelling ##################################
num=10
if Config['Flags']['Modelling']:

    for i in range(num):
        print(['data/XY_'+str(i)+'.mat'])
        temp = []
        #print(sio.loadmat(['data/XY_'+str(i)+'.mat']))
        temp = sio.loadmat('data/XY_'+str(i)+'.mat')
        X = temp['X']
        Y = temp['Y']
        index = (~np.isnan(X[:,7]) & (Y[:,1]>=1e4) & (X[:,0]>200))
        X_t = X[index,:]
        Y_t = Y[index,1]
        
        _ , _ = project.Dataset_create(X_t.T, Y_t.T)
        #Print_test(Out_train)
        #Print_test(Out_test)
        time.sleep(.5)

    # to make it simple, change index_Y.shape to 1

        #Print_test(len(index_Y))
        net = project.Modelling(12, 1)
        Print_test(net)
        time.sleep(.5)

['data/XY_0.mat']
> /export/scratch2/andong/Workspace/4D_Ne/Src/Py_Fun.py(278)Dataset_create()
    277         ipdb.set_trace()
--> 278         torch_dataset = Data.TensorDataset(X.T, Y.T)
    279         train_size = int(Config['NN_parameters']['Train_percent'] * X.shape[1])

ipdb> Y.shape
(741161,)
ipdb> n
TypeError: 'int' object is not callable
> /export/scratch2/andong/Workspace/4D_Ne/Src/Py_Fun.py(278)Dataset_create()
    277         ipdb.set_trace()
--> 278         torch_dataset = Data.TensorDataset(X.T, Y.T)
    279         train_size = int(Config['NN_parameters']['Train_percent'] * X.shape[1])



In [7]:
Y_t.shape

(741161,)